<a href="https://colab.research.google.com/github/andrealekoska/ohrid-sentiment-analysis/blob/master/Ohrid_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_excel('/content/DataOhrid_FINAL.xlsx')

# Convert star ratings to sentiment labels
def to_sentiment(rating):
    if rating >= 4:
        return 'positive'
    elif rating == 3:
        return 'neutral'
    else:
        return 'negative'

df['sentiment'] = df['rating'].apply(to_sentiment)

# Clean text: lowercase, remove non-letter characters
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['review_text'].apply(clean_text)

# Split into train (80%) and test (20%)
X = df['clean_text']
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Total reviews:", len(df))
print(df['sentiment'].value_counts())
print("Train:", len(X_train), "| Test:", len(X_test))

In [ ]:
# ===== Model 1: VADER (lexicon-based) =====
!pip install vaderSentiment -q
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, classification_report

analyzer = SentimentIntensityAnalyzer()

def vader_predict(text):
    score = analyzer.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

vader_preds = X_test.apply(vader_predict)
print("=== VADER ===")
print("Accuracy:", round(accuracy_score(y_test, vader_preds), 3))
print(classification_report(y_test, vader_preds, zero_division=0))

In [ ]:
# ===== Models 2 & 3: Naive Bayes and Logistic Regression =====
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

# Convert text to TF-IDF features
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Model 2: Naive Bayes
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_vec, y_train)
nb_preds = nb_model.predict(X_test_vec)
print("=== NAIVE BAYES ===")
print("Accuracy:", round(accuracy_score(y_test, nb_preds), 3))
print(classification_report(y_test, nb_preds, zero_division=0))

# Model 3: Logistic Regression
logreg = LogisticRegression(max_iter=1000, class_weight='balanced')
logreg.fit(X_train_vec, y_train)
logreg_preds = logreg.predict(X_test_vec)
print("=== LOGISTIC REGRESSION ===")
print("Accuracy:", round(accuracy_score(y_test, logreg_preds), 3))
print(classification_report(y_test, logreg_preds, zero_division=0))

In [ ]:
# ===== Model 4: RoBERTa (transformer-based) =====
!pip install transformers torch -q
from transformers import pipeline

roberta = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

def roberta_predict(text):
    result = roberta(text[:512])[0]['label']
    return result.lower()

roberta_preds = X_test.apply(roberta_predict)
print("=== ROBERTA ===")
print("Accuracy:", round(accuracy_score(y_test, roberta_preds), 3))
print(classification_report(y_test, roberta_preds, zero_division=0))

In [ ]:
# ===== 5-fold cross-validation for robustness =====
from sklearn.model_selection import cross_val_score

X_all_vec = vectorizer.fit_transform(X)

print("=== 5-FOLD CROSS-VALIDATION (Accuracy) ===\n")

nb_cv = cross_val_score(MultinomialNB(alpha=0.1), X_all_vec, y, cv=5, scoring='accuracy')
print(f"Naive Bayes:         {nb_cv.mean():.3f} (+/- {nb_cv.std():.3f})")

lr_cv = cross_val_score(LogisticRegression(max_iter=1000, class_weight='balanced'), X_all_vec, y, cv=5, scoring='accuracy')
print(f"Logistic Regression: {lr_cv.mean():.3f} (+/- {lr_cv.std():.3f})")

In [ ]:
# ===== Visualization of results =====
import matplotlib.pyplot as plt

models = ['VADER', 'Naive Bayes', 'Logistic Reg', 'RoBERTa']
accuracy = [0.59, 0.68, 0.71, 0.67]
neutral_f1 = [0.12, 0.31, 0.50, 0.26]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

ax1.bar(models, accuracy, color=colors)
ax1.set_title('Accuracy by Model (single split)', fontweight='bold')
ax1.set_ylabel('Accuracy')
ax1.set_ylim(0, 1)

ax2.bar(models, neutral_f1, color=colors)
ax2.set_title('F1-score for Neutral Class', fontweight='bold')
ax2.set_ylabel('F1-score')
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('/content/results_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved as results_chart.png")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models_cv = ['Naive Bayes', 'Logistic Reg']
single_split = [0.68, 0.71]
cross_val = [0.61, 0.60]

x = np.arange(len(models_cv))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, single_split, width, label='Single split', color='#45B7D1')
ax.bar(x + width/2, cross_val, width, label='5-fold CV', color='#FF6B6B')
ax.set_title('Single Split vs Cross-Validation', fontweight='bold')
ax.set_ylabel('Accuracy')
ax.set_xticks(x)
ax.set_xticklabels(models_cv)
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.savefig('/content/cv_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print(len(df))


In [ ]:
# ===== Secondary analysis: sentiment by domain =====
df['domain'] = df['source'].str.extract(r'\((\w+)\)')

domain_sentiment = df.groupby('domain')['sentiment'].value_counts(normalize=True).unstack() * 100
print("=== SENTIMENT BY DOMAIN (%) ===\n")
print(domain_sentiment.round(1))

In [ ]:
# ===== Chart: sentiment by domain =====
import matplotlib.pyplot as plt

domain_sentiment[['negative', 'neutral', 'positive']].plot(
    kind='bar', figsize=(9, 5),
    color=['#FF6B6B', '#FFD93D', '#6BCB77']
)
plt.title('Sentiment Distribution by Domain', fontweight='bold')
plt.ylabel('Percentage (%)')
plt.xlabel('Domain')
plt.xticks(rotation=0)
plt.legend(title='Sentiment')
plt.tight_layout()
plt.savefig('/content/domain_sentiment.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved as domain_sentiment.png")

In [ ]:
# ===== Confusion matrix for the best model (Logistic Regression) =====
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

labels = ['negative', 'neutral', 'positive']
cm = confusion_matrix(y_test, logreg_preds, labels=labels)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression', fontweight='bold')
plt.savefig('/content/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved as confusion_matrix.png")